# Entraînement du détecteur de cartes (GPU gratuit Colab)

1. Menu **Exécution → Modifier le type d'exécution → GPU (T4)**
2. *(Optionnel, pour affiner le modèle existant : plus rapide et meilleur)* glisser votre **best.pt** dans le panneau **Fichiers** à gauche (dossier `/content`).
3. **Exécution → Tout exécuter**. Gardez l'onglet ouvert et empêchez la mise en veille du PC.
4. À la fin, `cards.onnx`, `meta.json` et `best.pt` se téléchargent : déposer les deux premiers dans `app/model/` sur GitHub (branche `claude/adoring-gates-5by66o`), garder `best.pt` pour la prochaine fois.

In [ ]:
!nvidia-smi -L || echo 'PAS DE GPU : Exécution > Modifier le type d’exécution > GPU'

In [ ]:
BRANCH = 'claude/adoring-gates-5by66o'
!rm -rf cardcounter && git clone -q --depth 1 -b $BRANCH https://github.com/hugohismans/cardcounter.git
%cd cardcounter/detector
!pip -q install 'ultralytics>=8.4' opencv-python-headless onnx onnxslim onnxruntime

Paramètres : plus d'images / d'epochs = meilleur modèle, mais plus long.

In [ ]:
import os
# best.pt déposé dans /content -> on affine ce modèle (15 epochs), sinon on part de zéro (40 epochs)
FINETUNE = os.path.exists('/content/best.pt')
MODEL = '/content/best.pt' if FINETUNE else 'yolo26n.pt'
TRAIN, VAL, BATCH = 6000, 500, 32
EPOCHS = 15 if FINETUNE else 40
print('Affinage de best.pt' if FINETUNE else 'Entraînement depuis zéro', '-', EPOCHS, 'epochs')
os.environ.update(TRAIN=str(TRAIN), VAL=str(VAL), EPOCHS=str(EPOCHS), BATCH=str(BATCH), MODEL=MODEL)
!./train_all.sh

In [ ]:
from google.colab import files
files.download('../app/model/cards.onnx')
files.download('../app/model/meta.json')
files.download('runs/cards/weights/best.pt')  # utile pour affiner plus tard